In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Parameters
num_records = 1000

# Base data generation
customer_ids = [f"CUST{str(i).zfill(4)}" for i in range(1, 201)]
categories = ["Electronics", "Clothing", "Home & Garden", "Books", "Sports"]
countries = ["United States", "UK", "Canada", "Germany", "France", "Spain"]

data = []
for i in range(num_records):
    cust_id = random.choice(customer_ids)
    category = random.choice(categories)
    country = random.choice(countries)
    
    # Generate dates across the last year
    days_ago = random.randint(0, 365)
    date_val = datetime.now() - timedelta(days=days_ago)
    
    amount = round(random.uniform(10.0, 500.0), 2)
    loyalty_score = random.randint(1, 100)
    
    data.append({
        "TransactionID": f"TXN{str(i+1).zfill(5)}",
        "CustomerID": cust_id,
        "Date": date_val.strftime("%Y-%m-%d"),
        "ProductCategory": category,
        "Country": country,
        "TransactionAmount": amount,
        "LoyaltyScore": loyalty_score
    })

df = pd.DataFrame(data)

# Introduce 5% Missing Values
mask_amount = np.random.rand(num_records) < 0.05
df.loc[mask_amount, 'TransactionAmount'] = np.nan

mask_loyalty = np.random.rand(num_records) < 0.05
df.loc[mask_loyalty, 'LoyaltyScore'] = np.nan

# Introduce Erroneous transaction amounts (negative/extreme)
err_mask = np.random.rand(num_records) < 0.02
df.loc[err_mask, 'TransactionAmount'] = df.loc[err_mask, 'TransactionAmount'] * -1 # negative
err_outlier_mask = np.random.rand(num_records) < 0.01
df.loc[err_outlier_mask, 'TransactionAmount'] = df.loc[err_outlier_mask, 'TransactionAmount'] * 100 # outlier

# Inconsistent Date Formats
inc_date_mask = np.random.rand(num_records) < 0.1
df.loc[inc_date_mask, 'Date'] = df.loc[inc_date_mask, 'Date'].apply(
    lambda x: datetime.strptime(x, "%Y-%m-%d").strftime("%d-%b-%y")
)

# Inconsistent country names
messy_countries = {
    "United States": ["usa", "U.S.A.", "United States", "US"],
    "UK": ["United Kingdom", "uk", "U.K."],
    "Canada": ["canada", "CA"],
    "Germany": ["DE", "GER", "germany"],
    "France": ["FR", "france"],
    "Spain": ["ES", "spain"]
}

df['Country'] = df['Country'].apply(lambda x: random.choice(messy_countries.get(x, [x])))

# Invalid country names
invalid_country_mask = np.random.rand(num_records) < 0.02
df.loc[invalid_country_mask, 'Country'] = "InvalidCountry"

# Inconsistent ProductCategory cases
df['ProductCategory'] = df['ProductCategory'].apply(
    lambda x: x.lower() if random.random() < 0.3 else (x.upper() if random.random() < 0.3 else x)
)

# Inconsistent Capitalization in Customer IDs
df['CustomerID'] = df['CustomerID'].apply(
    lambda x: x.lower() if random.random() < 0.1 else x
)

# Generate 5% duplicates
num_duplicates = int(num_records * 0.05)
duplicates = df.sample(n=num_duplicates, random_state=42)
df = pd.concat([df, duplicates], ignore_index=True)

# Shuffle the dataframe
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save to CSV
df.to_csv("transactions_raw.csv", index=False)
print("Generated transactions_raw.csv with", len(df), "records.")
df.head()

In [ ]:
# Task 1: Data Profiling

import matplotlib.pyplot as plt
import seaborn as sns

# Check total number of records
print(f"Total number of records: {len(df)}")
print(f"Total number of columns: {len(df.columns)}")

# Check for duplicates
num_duplicates_found = df.duplicated().sum()
print(f"Number of duplicate rows: {num_duplicates_found}")

# Missing values summary
missing_summary = df.isnull().sum()
print("\nMissing values:\n", missing_summary[missing_summary > 0])

# Data distributions
plt.figure(figsize=(10, 5))
sns.histplot(df['TransactionAmount'].dropna(), bins=50, kde=True)
plt.title('Distribution of Transaction Amounts (Raw)')
plt.show()

# Data quality heatmap
plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.show()

# Range and average of transaction amounts
print("\nTransaction Amounts Description:")
print(df['TransactionAmount'].describe())

In [ ]:
# Task 2: Data Cleaning

# Remove duplicates
df_clean = df.drop_duplicates().copy()
print(f"Removed {len(df) - len(df_clean)} duplicates.")

# Fill Missing Values
avg_amount = df_clean['TransactionAmount'].mean()
df_clean['TransactionAmount'].fillna(avg_amount, inplace=True)

df_clean['LoyaltyScore'].fillna(50, inplace=True) # default score

# Remove or fix erroneous negative/extreme outliers for transaction amounts
print(f"Number of negative amounts: {len(df_clean[df_clean['TransactionAmount'] <= 0])}")
print(f"Number of extreme outliers (> 10000): {len(df_clean[df_clean['TransactionAmount'] > 10000])}")

# Let's fix negative amounts by taking absolute value
df_clean['TransactionAmount'] = df_clean['TransactionAmount'].abs()

# Cap outliers to a reasonable maximum (e.g., the 99th percentile or just removing them)
cap_value = 1000
df_clean.loc[df_clean['TransactionAmount'] > cap_value, 'TransactionAmount'] = cap_value

print(f"\nRemaining missing values after cleaning:\n{df_clean.isnull().sum()}")
print(f"Completeness improved: Total rows remaining: {len(df_clean)}")

In [ ]:
# Task 3: Data Standardization

# Function to parse dates consistently
def parse_date(date_str):
    try:
        return pd.to_datetime(date_str, format="%Y-%m-%d")
    except ValueError:
        try:
            return pd.to_datetime(date_str, format="%d-%b-%y")
        except ValueError:
            return pd.NaT

# 1. Dates
df_clean['Date'] = df_clean['Date'].apply(parse_date).dt.strftime("%Y-%m-%d")

# 2. Country names
country_mapping = {
    'usa': 'United States', 'U.S.A.': 'United States', 'US': 'United States',
    'United Kingdom': 'UK', 'uk': 'UK', 'U.K.': 'UK',
    'canada': 'Canada', 'CA': 'Canada',
    'DE': 'Germany', 'GER': 'Germany', 'germany': 'Germany',
    'FR': 'France', 'france': 'France',
    'ES': 'Spain', 'spain': 'Spain',
    'InvalidCountry': np.nan # drop or replace later
}

df_clean['Country'] = df_clean['Country'].replace(country_mapping)
df_clean = df_clean.dropna(subset=['Country']) # Drop invalid countries
df_clean['Country'] = df_clean['Country'].str.title()
# UK fix after title case
df_clean['Country'] = df_clean['Country'].replace('Uk', 'UK')

# 3. Product category names
df_clean['ProductCategory'] = df_clean['ProductCategory'].str.title()

# 4. Customer IDs capitalization
df_clean['CustomerID'] = df_clean['CustomerID'].str.upper()

print("Dates correctly formatted:\n", df_clean['Date'].head(3))
print("\nUnique Countries:\n", df_clean['Country'].unique())
print("\nUnique Product Categories:\n", df_clean['ProductCategory'].unique())

In [ ]:
# Task 4: Business Metrics & Aggregation

from IPython.display import display

# 1. Sales by Country
sales_by_country = df_clean.groupby('Country').agg(
    TotalSales=('TransactionAmount', 'sum'),
    AverageTransaction=('TransactionAmount', 'mean')
).reset_index().sort_values(by='TotalSales', ascending=False)
print("Sales by Country:")
display(sales_by_country)

# 2. Top 5 Customers
top_customers = df_clean.groupby('CustomerID').agg(
    TotalSpending=('TransactionAmount', 'sum')
).reset_index().sort_values(by='TotalSpending', ascending=False).head(5)
print("\nTop 5 Customers:")
display(top_customers)

# 3. Category Performance
category_perf = df_clean.groupby('ProductCategory').agg(
    TotalSales=('TransactionAmount', 'sum'),
    AverageTransaction=('TransactionAmount', 'mean')
).reset_index().sort_values(by='TotalSales', ascending=False)
print("\nCategory Performance:")
display(category_perf)

In [ ]:
# Task 5: Visualization & BI Reporting

# 1. Total Sales by Country
plt.figure(figsize=(10, 6))
sns.barplot(data=sales_by_country, x='Country', y='TotalSales', palette='coolwarm')
plt.title('Total Sales by Country')
plt.ylabel('Total Sales ($)')
plt.xticks(rotation=45)
plt.show()

# 2. Top 5 Customers
plt.figure(figsize=(8, 5))
sns.barplot(data=top_customers, x='CustomerID', y='TotalSpending', palette='viridis')
plt.title('Top 5 Customers by Spending')
plt.ylabel('Total Spending ($)')
plt.show()

# 3. Category Performance
plt.figure(figsize=(10, 6))
sns.barplot(data=category_perf, x='ProductCategory', y='TotalSales', palette='muted')
plt.title('Total Sales by Product Category')
plt.ylabel('Total Sales ($)')
plt.xticks(rotation=45)
plt.show()

# Export clean dataset
df_clean.to_csv('transactions_clean.csv', index=False)
print("Saved cleaned data to transactions_clean.csv")